# Q01 — Why do different bodies of water have different colours?

**Curriculum Level 1 (Observation)** · **Status:** `exploratory` (CPU-only notebook; configure local dataset paths in gitignored `config/local_paths.yaml` before executing).  
**Lesson shape:** **Predict → Reveal → Explain → Control → Limits** (adapted from the teaching-layer pattern of public `arrowfield` and the perceptual colour/key-gating conventions of public `espmap`).

### Step 0 — Predict (before running any code)
Before executing the cells below, write down your physical predictions:
1. Across GLORIA water-body types (`Coastal ocean`, `Lake`, `Estuary`, `River`), which will have the lowest Apparent Visible Wavelength (AVW) and highest Forel–Ule hue angle, and which will shift furthest toward yellow-brown?
2. Can two optically different waters (e.g., `Chl-dominated` vs. `Sediment-dominated`) share nearly the same hue angle?
3. When we compare a clear-ocean NASA PACE OCI swath (`PACE-01`) against GLORIA's global inland/coastal library, what fraction of satellite pixels do you expect to fall out-of-distribution (OOD) by Spectral Angle Mapper (SAM)?

### Scope & Controls
- **Controls:** Compare unflagged vs. QC-flagged counts, strict non-negative AVW vs. unclipped AVW (preserving negative $R_{rs}$ residuals rather than clipping to zero), and site-aggregated GLORIA medians (`Country|Site_name`) so replicate spectra do not compress internal OOD distances.
- **Colour convention:** Quantitative CIE 1931 2° chromaticity and display sRGB are used strictly for perceived water colour (`hydrospectral.color`); scalar geophysical maps (AVW) use Fabio Crameri's perceptually uniform, CVD-safe `batlow` colormap (`hydrospectral.palette`) gated by `render_key`.

In [ ]:
import os
from pathlib import Path

import numpy as np
import pandas as pd
import yaml

from hydrospectral.color import (
    chromaticity_to_hue_angle_deg,
    classify_owt_by_sam,
    rrs_to_xyz,
    xyz_to_chromaticity,
    xyz_to_display_srgb,
)
from hydrospectral.gloria import (
    build_site_group_keys,
    extract_rrs_matrix,
    load_gloria_tables,
    summarize_qc_flags,
)
from hydrospectral.manifests import load_active_sources
from hydrospectral.pace import (
    DEFAULT_PACE_EXCLUDE_FLAGS,
    build_rgb_composite,
    compute_swath_pca,
    decode_l2_flags,
    load_pace_l2_aop,
)
from hydrospectral.palette import compute_render_key, to_matplotlib_cmap
from hydrospectral.provenance import (
    CohortCounts,
    build_manifest,
    compute_cohort_key,
    inspect_git_provenance,
    sha256_file,
    write_manifest,
)
from hydrospectral.radiometry import compute_avw, compute_qwip_score
from hydrospectral.splits import nearest_neighbor_sam_ood

REPO_ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
SCRIPT_REL_PATH = 'notebooks/Q01_water_colour_and_owts.ipynb'
GIT_PROV = inspect_git_provenance(REPO_ROOT, SCRIPT_REL_PATH)
LOCAL_PATHS_FILE = REPO_ROOT / 'config' / 'local_paths.yaml'
assert LOCAL_PATHS_FILE.is_file(), (
    'Copy config/local_paths.example.yaml to config/local_paths.yaml and set '
    'datasets.gloria_2022_zip, datasets.pace_oci_20240730_nrt, and workspace.scratch_dir.'
)
local_cfg = yaml.safe_load(LOCAL_PATHS_FILE.read_text(encoding='utf-8'))
sources = load_active_sources(REPO_ROOT / 'manifests' / 'sources')

GLORIA_PATH = Path(local_cfg['datasets']['gloria_2022_zip'])
PACE_PATH = Path(local_cfg['datasets']['pace_oci_20240730_nrt'])
OUT_DIR = Path(
    os.environ.get(
        'Q01_OUT_DIR',
        str(Path(local_cfg['workspace']['scratch_dir']) / 'Q01_water_colour'),
    )
)
OUT_DIR.mkdir(parents=True, exist_ok=True)

WATER_BODY_TYPE_LABELS = {
    1: 'Lake',
    2: 'Estuary',
    3: 'Coastal ocean',
    4: 'River',
    5: 'Other',
}
WATER_TYPE_LABELS = {
    1: 'Sediment-dominated',
    2: 'Chl-dominated',
    3: 'CDOM-dominated',
    4: 'Chl+CDOM-dominated',
    5: 'Moderately turbid coastal',
    6: 'Clear',
}
print('Loaded active source manifests:', sorted(sources.keys()))
print('Output directory:', OUT_DIR)

## 1. GLORIA spectral quality, AVW, QWIP, and CIE 1931 chromaticity atlas

In [ ]:
gloria = load_gloria_tables(
    GLORIA_PATH,
    expected_sha256=sources['src.gloria-2022']['sha256'],
    include_qc_flags=True,
)
qc_summary = summarize_qc_flags(gloria.qc_flags)
wl_gloria, X_gloria = extract_rrs_matrix(gloria.rrs, min_nm=400, max_nm=700, step_nm=1)

complete_vis = np.isfinite(X_gloria).all(axis=1)
unflagged = ~qc_summary['any_flag'].to_numpy(dtype=bool)
n_total_gloria = len(gloria.meta)
n_incomplete_vis = int((~complete_vis).sum())
n_qc_flagged_complete = int((complete_vis & (~unflagged)).sum())
keep_vis = complete_vis & unflagged
X_vis = X_gloria[keep_vis]

# Preserve negative Rrs diagnostics rather than clipping.
has_neg_vis = np.any(X_vis < 0.0, axis=1)
n_neg_vis = int(has_neg_vis.sum())
avw_nm = compute_avw(X_vis, wl_gloria, require_nonnegative=True)
qwip = compute_qwip_score(X_vis, wl_gloria, require_nonnegative_avw=True)
xyz_e = rrs_to_xyz(X_vis, wl_gloria, illuminant='E')
xy_e = xyz_to_chromaticity(xyz_e)
hue_deg = chromaticity_to_hue_angle_deg(xy_e)
srgb = xyz_to_display_srgb(xyz_e, illuminant='E', exposure='per_spectrum_max', gamut='desaturate')

meta_vis = gloria.meta.loc[keep_vis].reset_index(drop=True)
wbt_codes = pd.to_numeric(meta_vis.get('Water_body_type'), errors='coerce')
wt_codes = pd.to_numeric(meta_vis.get('Water_type'), errors='coerce')
wbt_labels = wbt_codes.map(WATER_BODY_TYPE_LABELS).fillna('Unassigned')
wt_labels = wt_codes.map(WATER_TYPE_LABELS).fillna('Unassigned')

gloria_color = pd.DataFrame({
    'GLORIA_ID': meta_vis['GLORIA_ID'].astype(str).to_numpy(),
    'water_body_type': wbt_labels.to_numpy(dtype=str),
    'water_type': wt_labels.to_numpy(dtype=str),
    'has_negative_rrs': has_neg_vis,
    'avw_nm': avw_nm,
    'qwip_score': qwip['qwip_score'],
    'qwip_fail': qwip['qwip_fail'],
    'cie_x': xy_e[:, 0],
    'cie_y': xy_e[:, 1],
    'hue_angle_deg': hue_deg,
    'srgb_r': srgb[:, 0],
    'srgb_g': srgb[:, 1],
    'srgb_b': srgb[:, 2],
})


def _summarize_color_group(df: pd.DataFrame, group_col: str, order: list[str]) -> pd.DataFrame:
    rows = []
    for label in order + ['All']:
        sub = df if label == 'All' else df[df[group_col] == label]
        if sub.empty:
            continue
        avw_ok = sub['avw_nm'].dropna()
        hue_ok = sub['hue_angle_deg'].dropna()
        q_ok = sub['qwip_score'].dropna()
        rows.append({
            group_col: label,
            'n_spectra': int(len(sub)),
            'n_with_negative_rrs': int(sub['has_negative_rrs'].sum()),
            'n_finite_avw': int(len(avw_ok)),
            'n_finite_hue': int(len(hue_ok)),
            'n_qwip_fail': int((sub['qwip_fail'] == 1.0).sum()),
            'avw_median_nm': round(float(avw_ok.median()), 4) if len(avw_ok) else np.nan,
            'avw_p25_nm': round(float(avw_ok.quantile(0.25)), 4) if len(avw_ok) else np.nan,
            'avw_p75_nm': round(float(avw_ok.quantile(0.75)), 4) if len(avw_ok) else np.nan,
            'hue_median_deg': round(float(hue_ok.median()), 4) if len(hue_ok) else np.nan,
            'hue_p25_deg': round(float(hue_ok.quantile(0.25)), 4) if len(hue_ok) else np.nan,
            'hue_p75_deg': round(float(hue_ok.quantile(0.75)), 4) if len(hue_ok) else np.nan,
            'cie_x_median': round(float(sub['cie_x'].dropna().median()), 5) if sub['cie_x'].notna().any() else np.nan,
            'cie_y_median': round(float(sub['cie_y'].dropna().median()), 5) if sub['cie_y'].notna().any() else np.nan,
            'qwip_median': round(float(q_ok.median()), 5) if len(q_ok) else np.nan,
        })
    return pd.DataFrame(rows)


gloria_wbt_summary = _summarize_color_group(
    gloria_color,
    'water_body_type',
    ['Lake', 'Coastal ocean', 'Estuary', 'River', 'Other', 'Unassigned'],
)
gloria_wt_summary = _summarize_color_group(
    gloria_color,
    'water_type',
    [
        'Clear',
        'Moderately turbid coastal',
        'Chl-dominated',
        'Chl+CDOM-dominated',
        'CDOM-dominated',
        'Sediment-dominated',
        'Unassigned',
    ],
)

gloria_wbt_summary.to_csv(
    OUT_DIR / 'gloria_color_by_water_body_type.csv',
    index=False,
    encoding='utf-8',
    lineterminator='\n',
)
gloria_wt_summary.to_csv(
    OUT_DIR / 'gloria_color_by_water_type.csv',
    index=False,
    encoding='utf-8',
    lineterminator='\n',
)
print(
    f'Retained unflagged complete 400-700 nm GLORIA spectra: {len(gloria_color)} '
    f'(with >=1 negative Rrs: {n_neg_vis})'
)
print(gloria_wbt_summary.to_string(index=False))

## 2. PACE-01 Gulf of Mexico swath: flag screening, composites, PCA, AVW, and CIE colour

In [ ]:
# Optional: pass bbox=(-92.0, 26.0, -86.0, 30.5) to subset the northern Gulf swath on constrained RAM.
pace = load_pace_l2_aop(
    PACE_PATH,
    expected_sha256=sources['src.pace-oci-20240730-nrt']['sha256'],
)
active_exclude = tuple(f for f in DEFAULT_PACE_EXCLUDE_FLAGS if f in pace.flag_definitions)
skipped_flags = [f for f in DEFAULT_PACE_EXCLUDE_FLAGS if f not in pace.flag_definitions]
assert active_exclude, 'No recognized PACE l2_flags found in granule attributes; refusing unscreened swath.'
if skipped_flags:
    print('Flag names absent from granule attributes (skipped):', skipped_flags)

flag_res = decode_l2_flags(pace.l2_flags, pace.flag_definitions, exclude_flags=active_exclude)
valid_mask = flag_res['valid_mask']

rgb_vis, vis_wl = build_rgb_composite(
    pace.rrs, pace.wavelengths_nm, rgb_wavelengths_nm=(665.0, 550.0, 442.0), valid_mask=valid_mask
)
rgb_false, false_wl = build_rgb_composite(
    pace.rrs, pace.wavelengths_nm, rgb_wavelengths_nm=(704.0, 620.0, 550.0), valid_mask=valid_mask
)
pca = compute_swath_pca(
    pace.rrs, pace.wavelengths_nm, valid_mask=valid_mask, min_nm=400.0, max_nm=719.0, n_components=3
)

# Compute PACE trapezoidal AVW (PACE OCI wavelength_3d is non-uniform across ~600 nm) and CIE colour.
pace_ok = pca['valid_mask_used']
pace_valid_rrs = pace.rrs[pace_ok]
pace_vis_mask_full = (pace.wavelengths_nm >= 400.0) & (pace.wavelengths_nm <= 700.0)
pace_has_neg_vis = np.any(pace_valid_rrs[:, pace_vis_mask_full] < 0.0, axis=1)
pace_avw_nm = compute_avw(
    pace_valid_rrs, pace.wavelengths_nm, integration='trapezoid', require_nonnegative=True
)
pace_avw_unclipped_nm = compute_avw(
    pace_valid_rrs, pace.wavelengths_nm, integration='trapezoid', require_nonnegative=False
)
pace_xyz = rrs_to_xyz(pace_valid_rrs, pace.wavelengths_nm, illuminant='E')
pace_xy = xyz_to_chromaticity(pace_xyz)
pace_hue_deg = chromaticity_to_hue_angle_deg(pace_xy)
pace_srgb = xyz_to_display_srgb(pace_xyz, illuminant='E', exposure='max_y', gamut='desaturate')

print('Visible RGB wavelengths (nm):', vis_wl)
print('False-colour wavelengths (nm):', false_wl)
print('Swath PCA explained variance ratio (PC1-PC3):', np.round(pca['explained_variance_ratio'], 4))
print(
    f'PACE valid pixels: {int(pace_ok.sum())} | with >=1 neg Rrs (400-700 nm): {int(pace_has_neg_vis.sum())} '
    f'| finite AVW: {int(np.isfinite(pace_avw_nm).sum())} | finite CIE hue: {int(np.isfinite(pace_hue_deg).sum())}'
)
print('Swath pixel accounting:', pca['counts'].to_dict())

## 3. Exploratory GLORIA AVW-quartile optical archetypes, site-aggregated SAM OOD bridge, figures & manifest

We align non-negative GLORIA spectra onto PACE OCI 400–700 nm band centers via linear interpolation (exploratory alignment, not official OCI SRF convolution), build four quartile optical archetypes for `classify_owt_by_sam` (distinct from published Spyrakos 13-type or Bi & Hieronymi 10-type libraries), and aggregate GLORIA spectra by `Country|Site_name` median before `nearest_neighbor_sam_ood` so same-site replicates do not compress the internal LOO SAM threshold.

In [ ]:
pace_vis_mask = (pace.wavelengths_nm >= 400.0) & (pace.wavelengths_nm <= 700.0)
wl_common = pace.wavelengths_nm[pace_vis_mask]

# Filter to non-negative GLORIA spectra with valid AVW and site keys.
site_keys_all, site_valid_all = build_site_group_keys(gloria.meta)
site_keys_vis = site_keys_all[keep_vis]
site_valid_vis = site_valid_all[keep_vis]
gloria_ref_mask = np.isfinite(avw_nm) & (np.linalg.norm(X_vis, axis=1) > 0.0) & site_valid_vis
assert int(gloria_ref_mask.sum()) >= 4, 'Too few valid GLORIA reference spectra for optical archetypes.'

X_gloria_common = np.vstack([
    np.interp(wl_common, wl_gloria, row) for row in X_vis[gloria_ref_mask]
])
avw_ref = avw_nm[gloria_ref_mask]
sites_ref = site_keys_vis[gloria_ref_mask]

# Four exploratory AVW-quartile median archetypes on the PACE 400-700 nm grid.
q_edges = np.quantile(avw_ref, [0.0, 0.25, 0.50, 0.75, 1.0])
archetype_labels = ['Q1_bluest_AVW', 'Q2_blue_green_AVW', 'Q3_green_AVW', 'Q4_reddest_AVW']
archetype_spectra = []
archetype_train_counts = []
for i in range(4):
    lo_q, hi_q = q_edges[i], q_edges[i + 1]
    in_q = (avw_ref >= lo_q) & (avw_ref <= hi_q if i == 3 else avw_ref < hi_q)
    archetype_spectra.append(np.median(X_gloria_common[in_q], axis=0))
    archetype_train_counts.append(int(in_q.sum()))
archetype_matrix = np.vstack(archetype_spectra)

# Site-level median GLORIA spectra for replicate-safe LOO SAM OOD calibration.
site_df = pd.DataFrame(X_gloria_common).assign(site=sites_ref).groupby('site', sort=True).median()
X_gloria_site_medians = site_df.to_numpy(dtype=float)
assert X_gloria_site_medians.shape[0] >= 2, 'Need >= 2 distinct GLORIA sites for SAM OOD threshold.'

# Deterministic random sample of up to 2,000 valid PACE pixels for interactive screening.
pace_specs = pace_valid_rrs[:, pace_vis_mask]
rng = np.random.default_rng(2026)
n_sample = min(2000, pace_specs.shape[0])
sample_idx = rng.choice(pace_specs.shape[0], size=n_sample, replace=False)
pace_sample = pace_specs[sample_idx]
pace_sample_avw = pace_avw_nm[sample_idx]
pace_sample_hue = pace_hue_deg[sample_idx]

ood = nearest_neighbor_sam_ood(X_gloria_site_medians, pace_sample, ood_quantile=0.95)
owt = classify_owt_by_sam(
    pace_sample,
    archetype_matrix,
    archetype_labels,
    wavelengths_nm=wl_common,
    max_angle_rad=ood['threshold_sam_rad'],
)

# Summarize exploratory archetype assignments (retaining SAM angle & spectral magnitude).
owt_rows = []
assigned_arr = np.asarray(owt['assigned_label'], dtype=object)
angles_arr = np.asarray(owt['min_sam_rad'], dtype=float)
mags_arr = np.asarray(owt['magnitude'], dtype=float)
for i, label in enumerate(archetype_labels):
    hit = assigned_arr == label
    owt_rows.append({
        'archetype': label,
        'avw_min_nm': round(float(q_edges[i]), 4),
        'avw_max_nm': round(float(q_edges[i + 1]), 4),
        'n_gloria_train': archetype_train_counts[i],
        'n_pace_sampled': int(hit.sum()),
        'pace_sampled_fraction': round(float(hit.sum() / n_sample), 6),
        'median_sam_rad': round(float(np.nanmedian(angles_arr[hit])), 5) if hit.any() else np.nan,
        'median_spectral_area_sr_inv_nm': round(float(np.nanmedian(mags_arr[hit])), 6) if hit.any() else np.nan,
    })
uncl_hit = np.asarray(owt['is_unclassified'], dtype=bool)
owt_rows.append({
    'archetype': 'unclassified_ood',
    'avw_min_nm': np.nan,
    'avw_max_nm': np.nan,
    'n_gloria_train': 0,
    'n_pace_sampled': int(uncl_hit.sum()),
    'pace_sampled_fraction': round(float(uncl_hit.sum() / n_sample), 6),
    'median_sam_rad': round(float(np.nanmedian(angles_arr[uncl_hit])), 5) if uncl_hit.any() else np.nan,
    'median_spectral_area_sr_inv_nm': round(float(np.nanmedian(mags_arr[uncl_hit])), 6) if uncl_hit.any() else np.nan,
})
owt_summary_df = pd.DataFrame(owt_rows)
owt_summary_df.to_csv(
    OUT_DIR / 'owt_archetype_summary.csv',
    index=False,
    encoding='utf-8',
    lineterminator='\n',
)

# Summarize PACE-01 swath optical statistics.
avw_p = pace_avw_nm[np.isfinite(pace_avw_nm)]
avw_u = pace_avw_unclipped_nm[np.isfinite(pace_avw_unclipped_nm)]
hue_p = pace_hue_deg[np.isfinite(pace_hue_deg)]
pace_summary_rows = [
    {'metric': 'swath_pixels_total', 'value': int(pace.l2_flags.size), 'unit': 'pixels'},
    {'metric': 'swath_pixels_valid_pca', 'value': int(pace_ok.sum()), 'unit': 'pixels'},
    {'metric': 'swath_pixels_with_negative_rrs_400_700nm', 'value': int(pace_has_neg_vis.sum()), 'unit': 'pixels'},
    {'metric': 'swath_pixels_finite_avw_nonneg', 'value': int(avw_p.size), 'unit': 'pixels'},
    {'metric': 'swath_pixels_finite_avw_unclipped', 'value': int(avw_u.size), 'unit': 'pixels'},
    {'metric': 'swath_pixels_finite_hue', 'value': int(hue_p.size), 'unit': 'pixels'},
    {'metric': 'pca_pc1_explained_variance_ratio', 'value': round(float(pca['explained_variance_ratio'][0]), 6), 'unit': 'fraction'},
    {'metric': 'pca_pc2_explained_variance_ratio', 'value': round(float(pca['explained_variance_ratio'][1]), 6), 'unit': 'fraction'},
    {'metric': 'pca_pc3_explained_variance_ratio', 'value': round(float(pca['explained_variance_ratio'][2]), 6), 'unit': 'fraction'},
    {'metric': 'avw_nonneg_p05_nm', 'value': round(float(np.quantile(avw_p, 0.05)), 4), 'unit': 'nm'},
    {'metric': 'avw_nonneg_p25_nm', 'value': round(float(np.quantile(avw_p, 0.25)), 4), 'unit': 'nm'},
    {'metric': 'avw_nonneg_p50_nm', 'value': round(float(np.quantile(avw_p, 0.50)), 4), 'unit': 'nm'},
    {'metric': 'avw_nonneg_p75_nm', 'value': round(float(np.quantile(avw_p, 0.75)), 4), 'unit': 'nm'},
    {'metric': 'avw_nonneg_p95_nm', 'value': round(float(np.quantile(avw_p, 0.95)), 4), 'unit': 'nm'},
    {'metric': 'avw_unclipped_p50_nm', 'value': round(float(np.quantile(avw_u, 0.50)), 4), 'unit': 'nm'},
    {'metric': 'hue_p25_deg', 'value': round(float(np.quantile(hue_p, 0.25)), 4), 'unit': 'deg'},
    {'metric': 'hue_p50_deg', 'value': round(float(np.quantile(hue_p, 0.50)), 4), 'unit': 'deg'},
    {'metric': 'hue_p75_deg', 'value': round(float(np.quantile(hue_p, 0.75)), 4), 'unit': 'deg'},
    {'metric': 'cie_x_median', 'value': round(float(np.nanmedian(pace_xy[:, 0])), 5), 'unit': 'chromaticity'},
    {'metric': 'cie_y_median', 'value': round(float(np.nanmedian(pace_xy[:, 1])), 5), 'unit': 'chromaticity'},
    {'metric': 'gloria_site_medians_count', 'value': int(len(X_gloria_site_medians)), 'unit': 'sites'},
    {'metric': 'sam_ood_threshold_95pct_rad', 'value': round(float(ood['threshold_sam_rad']), 6), 'unit': 'rad'},
    {'metric': 'pace_sampled_pixels', 'value': int(n_sample), 'unit': 'pixels'},
    {'metric': 'pace_sampled_ood_flagged', 'value': int(ood['is_ood'].sum()), 'unit': 'pixels'},
    {'metric': 'pace_sampled_ood_fraction', 'value': round(float(ood['is_ood'].mean()), 6), 'unit': 'fraction'},
]
for row in pace_summary_rows:
    v = row['value']
    row['value'] = str(int(v)) if isinstance(v, (int, np.integer)) else str(v)
pace_summary_df = pd.DataFrame(pace_summary_rows)
pace_summary_df.to_csv(
    OUT_DIR / 'pace_swath_summary.csv',
    index=False,
    encoding='utf-8',
    lineterminator='\n',
)

print(
    f'Site-aggregated GLORIA ({len(X_gloria_site_medians)} sites) 95th-pct LOO SAM threshold: '
    f'{ood["threshold_sam_rad"]:.4f} rad | PACE sampled OOD: {int(ood["is_ood"].sum())}/{n_sample}'
)
print(owt_summary_df.to_string(index=False))

# Optional figures when matplotlib (viz extra) is available.
try:
    import matplotlib
    matplotlib.use('Agg')
    import matplotlib.pyplot as plt
    import colour

    # Figure 1: GLORIA CIE 1931 chromaticity + AVW vs Forel-Ule hue angle
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(11.5, 5.0), dpi=150)

    cmfs = colour.MSDS_CMFS['CIE 1931 2 Degree Standard Observer']
    locus_wl = np.arange(380, 701, 5, dtype=float)
    locus_xyz = np.array([cmfs[w] for w in locus_wl], dtype=float)
    locus_xy = locus_xyz[:, :2] / locus_xyz.sum(axis=1, keepdims=True)
    locus_closed = np.vstack([locus_xy, locus_xy[0]])
    ax1.plot(locus_closed[:, 0], locus_closed[:, 1], color='#333333', lw=1.2, zorder=1)
    ax1.fill(locus_closed[:, 0], locus_closed[:, 1], color='#f7f7f7', zorder=0)

    ok_xy = np.isfinite(xy_e).all(axis=1) & np.isfinite(srgb).all(axis=1)
    ax1.scatter(
        xy_e[ok_xy, 0],
        xy_e[ok_xy, 1],
        c=np.clip(srgb[ok_xy], 0.0, 1.0),
        s=7,
        alpha=0.65,
        edgecolors='none',
        zorder=2,
    )
    ax1.scatter([1.0 / 3.0], [1.0 / 3.0], marker='+', s=90, color='black', lw=1.5, zorder=4, label='White point E (1/3, 1/3)')
    wbt_colors = {'Lake': '#1b9e77', 'Coastal ocean': '#2b8cbe', 'Estuary': '#d95f02', 'River': '#7570b3'}
    for wbt_name, col_hex in wbt_colors.items():
        row_s = gloria_wbt_summary[gloria_wbt_summary['water_body_type'] == wbt_name]
        if not row_s.empty:
            ax1.scatter(
                [row_s['cie_x_median'].iloc[0]],
                [row_s['cie_y_median'].iloc[0]],
                s=75,
                color=col_hex,
                edgecolors='white',
                linewidths=1.0,
                zorder=5,
                label=f'{wbt_name} median (n={int(row_s["n_spectra"].iloc[0])})',
            )
    ax1.set_xlim(0.0, 0.75)
    ax1.set_ylim(0.0, 0.85)
    ax1.set_xlabel('CIE 1931 x')
    ax1.set_ylabel('CIE 1931 y')
    ax1.set_title('GLORIA CIE 1931 2° Chromaticity (Illuminant E)')
    ax1.legend(loc='upper right', fontsize=8, framealpha=0.92)
    ax1.grid(alpha=0.25)

    for wbt_name, col_hex in wbt_colors.items():
        sub = gloria_color[(gloria_color['water_body_type'] == wbt_name) & gloria_color['avw_nm'].notna() & gloria_color['hue_angle_deg'].notna()]
        ax2.scatter(sub['avw_nm'], sub['hue_angle_deg'], s=8, alpha=0.35, color=col_hex, edgecolors='none', label=f'GLORIA {wbt_name}')
    ok_ps = np.isfinite(pace_sample_avw) & np.isfinite(pace_sample_hue)
    ax2.scatter(
        pace_sample_avw[ok_ps],
        pace_sample_hue[ok_ps],
        s=10,
        alpha=0.45,
        facecolors='none',
        edgecolors='#111111',
        linewidths=0.5,
        label=f'PACE-01 sample (n={int(ok_ps.sum())})',
    )
    ax2.set_xlabel('Apparent Visible Wavelength, AVW (nm)')
    ax2.set_ylabel('Forel-Ule Hue Angle alpha (deg)')
    ax2.set_title('AVW vs. Hue Angle: GLORIA vs. PACE-01 Sample')
    ax2.legend(loc='upper right', fontsize=8, framealpha=0.92)
    ax2.grid(alpha=0.25)

    fig.tight_layout()
    fig.savefig(OUT_DIR / 'fig01_gloria_chromaticity_and_avw.png', dpi=150, metadata={'Software': 'hydrospectral'})
    plt.close(fig)

    # Figure 2: PACE-01 4-panel optical overview cropped to 1st-99th pct valid-pixel extent
    ys, xs = np.nonzero(pace_ok)
    y0 = max(0, int(np.quantile(ys, 0.01)) - 20)
    y1 = min(pace_ok.shape[0], int(np.quantile(ys, 0.99)) + 20)
    x0 = max(0, int(np.quantile(xs, 0.01)) - 20)
    x1 = min(pace_ok.shape[1], int(np.quantile(xs, 0.99)) + 20)
    step = 2

    def _fill_bg(img_rgb: np.ndarray, bg: tuple[float, float, float] = (0.07, 0.09, 0.12)) -> np.ndarray:
        out = img_rgb[y0:y1:step, x0:x1:step].copy()
        bad = ~np.isfinite(out).all(axis=2)
        out[bad] = bg
        return np.clip(out, 0.0, 1.0)

    avw_grid = np.full(pace_ok.shape, np.nan, dtype=float)
    avw_grid[pace_ok] = pace_avw_nm
    srgb_grid = np.full(pace_ok.shape + (3,), np.nan, dtype=float)
    srgb_grid[pace_ok] = pace_srgb

    fig2, axes = plt.subplots(2, 2, figsize=(10.5, 9.0), dpi=150)
    axes[0, 0].imshow(_fill_bg(rgb_vis), aspect='auto')
    axes[0, 0].set_title(f'PACE-01 Visible RGB ({vis_wl[0]:.0f}, {vis_wl[1]:.0f}, {vis_wl[2]:.0f} nm)')
    axes[0, 1].imshow(_fill_bg(rgb_false), aspect='auto')
    axes[0, 1].set_title(f'PACE-01 False Colour ({false_wl[0]:.0f}, {false_wl[1]:.0f}, {false_wl[2]:.0f} nm)')

    avw_crop = avw_grid[y0:y1:step, x0:x1:step]
    cmap_avw = to_matplotlib_cmap('batlow', bad_color='#12171f')
    im = axes[1, 0].imshow(avw_crop, vmin=440.0, vmax=570.0, cmap=cmap_avw, aspect='auto')
    axes[1, 0].set_title('AVW (400-700 nm, non-neg Rrs; Crameri batlow)', fontsize=10.5)
    fig2.colorbar(im, ax=axes[1, 0], fraction=0.046, pad=0.04, label='AVW (nm)')

    axes[1, 1].imshow(_fill_bg(srgb_grid), aspect='auto')
    axes[1, 1].set_title('CIE 1931 2° Display sRGB Water Colour (Illuminant E)')

    for ax in axes.ravel():
        ax.set_xticks([])
        ax.set_yticks([])
    fig2.suptitle('PACE-01 (2024-07-30 V2.0 NRT) Optical Water Colour Overview (1st-99th pct valid swath extent)', fontsize=11)
    fig2.tight_layout()
    fig2.savefig(OUT_DIR / 'fig02_pace01_optical_overview.png', dpi=150, metadata={'Software': 'hydrospectral'})
    plt.close(fig2)
    print('Saved figures: fig01_gloria_chromaticity_and_avw.png, fig02_pace01_optical_overview.png')
except ImportError:
    print('matplotlib not installed; skipping PNG figures (install viz extra to render figures).')

gloria_counts = CohortCounts(
    attempted=n_total_gloria,
    completed=int(keep_vis.sum()),
    failed=0,
    excluded=int(n_total_gloria - keep_vis.sum()),
    exclusion_reasons={
        'incomplete_400_700nm': n_incomplete_vis,
        'qc_flagged': n_qc_flagged_complete,
    },
)
csv_names = [
    'gloria_color_by_water_body_type.csv',
    'gloria_color_by_water_type.csv',
    'owt_archetype_summary.csv',
    'pace_swath_summary.csv',
]
inputs_map = {
    'src.gloria-2022': gloria.archive_sha256,
    'src.pace-oci-20240730-nrt': pace.file_sha256,
}
q01_cohort_key = compute_cohort_key(
    inputs_sha256=inputs_map,
    target_col=None,
    wavelength_window_nm=[400.0, 700.0],
    qc_policy={
        'gloria_require_unflagged': True,
        'gloria_clip_negative_rrs': False,
        'pace_exclude_flags': list(active_exclude),
    },
    split_spec={'ood_aggregation': 'Country|Site_name_median', 'ood_quantile': 0.95, 'pace_sample_n': int(n_sample)},
    srf_spec='exploratory_linear_interp_pace_band_centers',
)
q01_render_key = compute_render_key(
    'batlow', 440.0, 570.0, kind='sequential', bad_color='#12171f', extra={'panel': 'pace01_avw_nonneg'}
)
manifest = build_manifest(
    experiment_id='Q01',
    title='GLORIA CIE water colour atlas and PACE-01 optical exploration',
    status='exploratory',
    git_commit=GIT_PROV['git_commit'],
    git_tree_clean=GIT_PROV['git_tree_clean'],
    cohort_key=q01_cohort_key,
    render_key=q01_render_key,
    script_path=SCRIPT_REL_PATH,
    script_sha256=sha256_file(REPO_ROOT / SCRIPT_REL_PATH),
    inputs=inputs_map,
    counts={'gloria_visible': gloria_counts, 'pace_swath_pca': pca['counts']},
    units={
        'Rrs': 'sr^-1',
        'avw_nm': 'nm',
        'hue_angle_deg': 'deg',
        'sam_rad': 'rad',
        'spectral_area': 'sr^-1 nm',
    },
    random_seed=2026,
    pairing_rule='Exploratory linear interpolation of site-median GLORIA spectra onto PACE OCI 400-700 nm band centers (not official OCI SRF convolution)',
    prior_exposure='Prior exploratory GLORIA audit and PACE-01 RGB/false-colour/PCA inspection documented in docs/aquatic-hyperspectral-handoff-2026-11-25.md (§4, §7)',
    extra={
        'n_gloria_site_medians': int(len(X_gloria_site_medians)),
        'n_pace_ood_sampled': int(n_sample),
        'n_pace_ood_flagged': int(ood['is_ood'].sum()),
        'sam_ood_threshold_rad': round(float(ood['threshold_sam_rad']), 6),
        'avw_colormap': 'crameri_batlow',
        'script_in_commit': bool(GIT_PROV['script_in_commit']),
        'output_csv_sha256': {name: sha256_file(OUT_DIR / name) for name in csv_names},
    },
)
write_manifest(manifest, OUT_DIR / 'manifest.json')
print('Wrote deterministic manifest to:', OUT_DIR / 'manifest.json')

## 4. Limits — What You May and May Not Conclude

| You may conclude | You may not conclude from water colour / OWTs alone |
| --- | --- |
| Relative shifts in visible spectral shape (AVW, CIE chromaticity, hue angle) across water-body types and across a satellite swath | Taxonomic identity, microbial community composition, or species dominance |
| Whether a satellite swath falls inside or outside an in situ spectral library's applicability domain (SAM OOD distance) | That a model calibrated on GLORIA inland/coastal spectra transfers reliably to clear open-ocean pixels or unrepresented tropical rivers |
| How preserving vs. clipping negative $R_{rs}$ residuals changes valid pixel counts and median AVW on NRT retrievals | That V2.0 NRT reflectance or linear band-center interpolation equals official SRF-convolved final science reprocessing |
| How four coarse GLORIA AVW-quartile prototypes behave as an exploratory baseline | Equivalence to published reference OWT libraries (Spyrakos et al. 2018; Bi & Hieronymi 2024) |